In [1]:
# ============================================================
# HOMEWORK 4.1 - PART II
# BARENBLATT PROFILE COEFFICIENT VERIFICATION
#
# Verify:
#
# k = ((m-1)/(2m)) * beta
#
# for
#
# rho_t = Delta(rho^m)
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# SIMILARITY EXPONENTS
# ============================================================

def similarity_exponents(
    d,
    m
):

    denominator = (
        d * (m - 1.0)
        +
        2.0
    )

    beta = (
        1.0
        /
        denominator
    )

    alpha = (
        d * beta
    )

    return (
        alpha,
        beta
    )


# ============================================================
# CORRECT k
# ============================================================

def correct_k(
    d,
    m
):

    alpha, beta = similarity_exponents(
        d,
        m
    )

    k = (

        (m - 1.0)

        /

        (2.0 * m)

        *

        beta
    )

    return (
        alpha,
        beta,
        k
    )


# ============================================================
# ANALYTIC SIMILARITY-EQUATION RESIDUAL
#
# We verify
#
# -alpha F - beta xi F'
#
# =
#
# (F^m)'' + ((d-1)/xi)(F^m)'
#
# ============================================================

def similarity_residual(
    xi,
    d,
    m,
    C,
    k
):

    alpha, beta = similarity_exponents(
        d,
        m
    )

    xi = np.asarray(
        xi,
        dtype=float
    )


    inside = (

        C
        -
        k * xi**2
    )


    if np.any(
        inside <= 0
    ):

        raise ValueError(
            "All xi values must lie strictly "
            "inside the Barenblatt support."
        )


    p = (
        1.0
        /
        (m - 1.0)
    )


    q = (
        m
        /
        (m - 1.0)
    )


    F = (
        inside**p
    )


    # --------------------------------------------------------
    # F'
    # --------------------------------------------------------

    F_prime = (

        -2.0
        *
        k
        *
        p
        *
        xi
        *
        inside**(p - 1.0)
    )


    # --------------------------------------------------------
    # LEFT SIDE
    #
    # -alpha F - beta xi F'
    # --------------------------------------------------------

    lhs = (

        -alpha * F

        -

        beta
        *
        xi
        *
        F_prime
    )


    # --------------------------------------------------------
    # Derivatives of F^m
    # --------------------------------------------------------

    Fm_prime = (

        -2.0
        *
        k
        *
        q
        *
        xi
        *
        inside**(q - 1.0)
    )


    Fm_second = (

        -2.0
        *
        k
        *
        q
        *
        inside**(q - 1.0)

        +

        4.0
        *
        k**2
        *
        q
        *
        (q - 1.0)
        *
        xi**2
        *
        inside**(q - 2.0)
    )


    # --------------------------------------------------------
    # RADIAL LAPLACIAN
    #
    # (F^m)''
    #
    # +
    #
    # (d-1)/xi * (F^m)'
    #
    # Handle xi = 0 using the limiting expression.
    # --------------------------------------------------------

    rhs = np.empty_like(
        xi
    )


    nonzero = (

        np.abs(xi)
        >
        1e-12
    )


    rhs[
        nonzero
    ] = (

        Fm_second[
            nonzero
        ]

        +

        (
            (d - 1.0)

            /

            xi[
                nonzero
            ]
        )

        *

        Fm_prime[
            nonzero
        ]
    )


    # At xi = 0:
    #
    # ((d-1)/xi) (F^m)'
    #
    # has limit
    #
    # -2kq(d-1) C^(q-1)
    #
    # so the full radial Laplacian is
    #
    # -2kqd C^(q-1)

    zero = ~nonzero


    rhs[
        zero
    ] = (

        -2.0
        *
        k
        *
        q
        *
        d
        *
        inside[
            zero
        ]**(q - 1.0)
    )


    residual = (

        lhs

        -

        rhs
    )


    return (
        lhs,
        rhs,
        residual
    )


# ============================================================
# TEST SEVERAL VALUES OF d AND m
# ============================================================

test_cases = [

    (1, 2.0),
    (1, 3.0),
    (2, 2.0),
    (2, 3.0),
    (3, 2.0),
    (3, 4.0),
    (5, 2.5)

]


results = []


C = 2.0


for d, m in test_cases:

    alpha, beta, k = correct_k(
        d,
        m
    )


    # --------------------------------------------------------
    # Radius of the similarity support:
    #
    # C - k xi^2 > 0
    #
    # xi < sqrt(C/k)
    # --------------------------------------------------------

    support_radius = np.sqrt(
        C / k
    )


    # Stay safely inside the free boundary

    xi = np.linspace(
        0.0,
        0.8 * support_radius,
        1001
    )


    lhs, rhs, residual = similarity_residual(

        xi=xi,

        d=d,

        m=m,

        C=C,

        k=k
    )


    max_abs_residual = np.max(

        np.abs(
            residual
        )
    )


    scale = max(

        np.max(
            np.abs(lhs)
        ),

        np.max(
            np.abs(rhs)
        ),

        1e-15
    )


    relative_residual = (

        max_abs_residual

        /

        scale
    )


    results.append({

        "d":
            d,

        "m":
            m,

        "alpha":
            alpha,

        "beta":
            beta,

        "k":
            k,

        "Support_Radius_Xi":
            support_radius,

        "Max_Absolute_Residual":
            max_abs_residual,

        "Relative_Residual":
            relative_residual

    })


results_df = pd.DataFrame(
    results
)


print("=" * 80)

print(
    "BARENBLATT COEFFICIENT VERIFICATION"
)

print("=" * 80)

print()

print(
    results_df.to_string(
        index=False
    )
)


# ============================================================
# AUTOMATIC CHECK
# ============================================================

max_residual = np.max(

    results_df[
        "Max_Absolute_Residual"
    ]
)


max_relative_residual = np.max(

    results_df[
        "Relative_Residual"
    ]
)


print()

print("=" * 80)

print(
    "AUTOMATIC VERIFICATION"
)

print("=" * 80)

print(
    "Maximum absolute PDE residual =",
    max_residual
)

print(
    "Maximum relative PDE residual =",
    max_relative_residual
)


if max_relative_residual < 1e-10:

    print()

    print(
        "PASS: The proposed k satisfies the "
        "similarity equation for every test case."
    )

else:

    print()

    print(
        "WARNING: Residual is larger than expected."
    )


# ============================================================
# EXAMPLE:
# d = 1, m = 2
# ============================================================

d_example = 1

m_example = 2.0


alpha, beta, k = correct_k(

    d_example,

    m_example
)


print()

print("=" * 80)

print(
    "EXAMPLE: d = 1, m = 2"
)

print("=" * 80)

print(
    "alpha =",
    alpha
)

print(
    "beta =",
    beta
)

print(
    "k =",
    k
)

print()

print(
    "Expected alpha = 1/3 =",
    1.0 / 3.0
)

print(
    "Expected beta = 1/3 =",
    1.0 / 3.0
)

print(
    "Expected k = 1/12 =",
    1.0 / 12.0
)


# ============================================================
# TEST COMMON INCORRECT COEFFICIENTS
#
# This is useful because the homework explicitly warns that
# LLMs often get this coefficient wrong.
# ============================================================

print()

print("=" * 80)

print(
    "CORRECT k VS. COMMON INCORRECT COEFFICIENTS"
)

print("=" * 80)


d = 2

m = 3.0


alpha, beta = similarity_exponents(
    d,
    m
)


candidate_k = {

    "Correct":
        (
            (m - 1.0)
            /
            (2.0 * m)
            *
            beta
        ),

    "Missing (m-1)":
        (
            beta
            /
            (2.0 * m)
        ),

    "Missing m":
        (
            (m - 1.0)
            /
            2.0
            *
            beta
        ),

    "Missing factor 2":
        (
            (m - 1.0)
            /
            m
            *
            beta
        ),

    "Just beta":
        beta

}


candidate_results = []


for name, k_test in candidate_k.items():

    support_radius = np.sqrt(
        C / k_test
    )


    xi = np.linspace(
        0.0,
        0.7 * support_radius,
        1001
    )


    lhs, rhs, residual = similarity_residual(

        xi=xi,

        d=d,

        m=m,

        C=C,

        k=k_test
    )


    max_abs_residual = np.max(

        np.abs(
            residual
        )
    )


    scale = max(

        np.max(
            np.abs(lhs)
        ),

        np.max(
            np.abs(rhs)
        ),

        1e-15
    )


    relative_residual = (

        max_abs_residual

        /

        scale
    )


    candidate_results.append({

        "Coefficient":
            name,

        "k":
            k_test,

        "Max_Absolute_Residual":
            max_abs_residual,

        "Relative_Residual":
            relative_residual

    })


candidate_df = pd.DataFrame(
    candidate_results
)


print()

print(
    candidate_df.to_string(
        index=False
    )
)


# ============================================================
# VERIFY CORRECT COEFFICIENT IS THE ONLY ONE WITH
# NEAR-ZERO RESIDUAL
# ============================================================

correct_residual = candidate_df.loc[

    candidate_df[
        "Coefficient"
    ]
    ==
    "Correct",

    "Relative_Residual"

].iloc[0]


incorrect_residuals = candidate_df.loc[

    candidate_df[
        "Coefficient"
    ]
    !=
    "Correct",

    "Relative_Residual"

]


print()

print("=" * 80)

print(
    "FINAL VERIFICATION"
)

print("=" * 80)

print(
    "Correct coefficient relative residual =",
    correct_residual
)

print(
    "Smallest incorrect coefficient residual =",
    incorrect_residuals.min()
)


if (
    correct_residual < 1e-10

    and

    np.all(
        incorrect_residuals > 1e-4
    )
):

    print()

    print(
        "PASS: k = ((m-1)/(2m)) beta "
        "satisfies the porous medium equation."
    )

    print(
        "PASS: The tested incorrect coefficients "
        "do not satisfy the equation."
    )


# ============================================================
# FINAL FORMULA
# ============================================================

print()

print("=" * 80)

print(
    "FINAL FORMULA"
)

print("=" * 80)

print()

print(
    "k = ((m - 1) / (2m)) * beta"
)

print()

print(
    "beta = 1 / [d(m-1) + 2]"
)

print()

print(
    "Therefore:"
)

print(
    "k = (m - 1) / {2m[d(m-1) + 2]}"
)

BARENBLATT COEFFICIENT VERIFICATION

 d   m    alpha     beta        k  Support_Radius_Xi  Max_Absolute_Residual  Relative_Residual
 1 2.0 0.333333 0.333333 0.083333           4.898979           2.220446e-16       3.330669e-16
 1 3.0 0.250000 0.250000 0.083333           4.898979           5.551115e-17       1.570092e-16
 2 2.0 0.500000 0.250000 0.062500           5.656854           1.110223e-16       1.110223e-16
 2 3.0 0.333333 0.166667 0.055556           6.000000           1.110223e-16       2.355139e-16
 3 2.0 0.600000 0.200000 0.050000           6.324555           4.440892e-16       3.700743e-16
 3 4.0 0.272727 0.090909 0.034091           7.659417           1.665335e-16       4.846515e-16
 5 2.5 0.526316 0.105263 0.031579           7.958224           3.330669e-16       3.986561e-16

AUTOMATIC VERIFICATION
Maximum absolute PDE residual = 4.440892098500626e-16
Maximum relative PDE residual = 4.846515292325547e-16

PASS: The proposed k satisfies the similarity equation for every test 